# v5: continual run to 30 cycles (GPU 0)
Hindi → Chinese → Arabic, repeated. Phases 19-90 continue the v4b run.
* **Same recipe as v4b**: same model (QK-norm), same per-phase schedule (fresh AdamW, warmup, stable until plateau, cooldown), same divergence guard (val > 1.15x best twice → retry at half the learning rate, max 2), z-loss off, script-restricted loss and decoding.
* **Resumes** from the last saved phase (phase 18) and continues to **phase 90 = 30 cycles**. Nothing restarts from scratch.
* **Every phase** saves a model-only checkpoint `runs/<run>/ckpt/pXXX_<lang>.pt` (~270 MB) and appends to `phases.jsonl` / `log.jsonl`.
* **Best models**: `runs/<run>/best/best_<lang>.pt` = the checkpoint with the highest chrF on that language (`best.json` says which phase).
* **HF backup**: each new checkpoint, the logs and any new best model are uploaded to your HF repo in the background, so a dead instance loses at most one phase.
* Value vectors are hookable (`probe_points(model)` in common_v5.py): used later by the analysis, not during training.
* Rerunning the notebook at any time resumes where it stopped.

In [1]:
import os, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"      # run this cell FIRST in a fresh kernel
REPO = "/workspace/shreshth"
os.chdir(REPO); sys.path.insert(0, REPO)
from common_v5 import *
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")

device: cuda | NVIDIA GeForce RTX 4090


In [2]:
# Get everything this run needs onto this machine (skips what is already here)
from huggingface_hub import HfApi, snapshot_download
RUN = "v4b_continual_s0"
HF_REPO = HfApi().whoami()["name"] + "/cml-lang-self-checkpoints"

# 1) the run so far (checkpoints p000-p018 + logs), from the HF backup
if not glob.glob(P("runs", RUN, "ckpt", "p*.pt")):
    snapshot_download(HF_REPO, local_dir=P("runs"), allow_patterns=[f"{RUN}/*"])
ck = sorted(glob.glob(P("runs", RUN, "ckpt", "p*.pt")))
print(f"{RUN}: {len(ck)} checkpoints, last = {os.path.basename(ck[-1]) if ck else None}")

# 2) FLORES+ (gated: needs your HF login)
FLORES_CODE = {"en": "eng_Latn", "hi": "hin_Deva", "zh": "cmn_Hans", "ar": "arb_Arab"}
for l, code in FLORES_CODE.items():
    for split in ["dev", "devtest"]:
        p = P(f"data/flores/{split}.{l}.jsonl")
        if not os.path.exists(p):
            from datasets import load_dataset
            os.makedirs(os.path.dirname(p), exist_ok=True)
            ds = load_dataset("openlanguagedata/flores_plus", code, split=split)
            with open(p, "w", encoding="utf-8") as f:
                for r in ds:
                    f.write(json.dumps({"text": r["text"]}, ensure_ascii=False) + "\n")
            print("downloaded", p, len(ds))

# 3) training data: must already be here
missing = [f"data/en-{l}/{s}.jsonl" for l in LANGS for s in ["train", "val"] if not os.path.exists(P(f"data/en-{l}/{s}.jsonl"))]
print("training data OK" if not missing else f"MISSING: {missing}  -> copy them here before training (Arabic: arabic.ipynb rebuilds it)")

v4b_continual_s0: 19 checkpoints, last = p018_ar.pt
training data OK


In [3]:
C = RunConfig(run_name=RUN, mode="continual", order=("hi", "zh", "ar"), n_phases=90, seed=0,
              time_cap_h=120.0, hf_repo=HF_REPO)
print(C)
nxt = max(int(os.path.basename(p)[1:4]) for p in ck) + 1 if ck else 0
print(f"will resume at phase {nxt} ({phase_lang(C, nxt) if nxt else 'init'}) and stop after phase {C.n_phases}")

RunConfig(run_name='v4b_continual_s0', mode='continual', order=('hi', 'zh', 'ar'), init_ckpt=None, n_phases=90, seed=0, max_tokens=16000, lr=0.0002, min_lr=2e-05, warmup_steps=500, cooldown_steps=2000, min_steps=2000, max_stable_steps=60000, eval_every=500, plateau_window=4, plateau_tol=0.005, weight_decay=0.1, grad_clip=1.0, masked=True, diverge_factor=1.15, max_retries=2, time_cap_h=120.0, hf_repo='s8n29/cml-lang-self-checkpoints')
will resume at phase 19 (hi) and stop after phase 90


In [4]:
D = load_data(train_langs=["hi", "zh", "ar"])

  hi: 993,016 train / 2,000 val
  zh: 755,626 train / 2,000 val
  ar: 998,000 train / 2,000 val
data loaded in 51s


In [ ]:
model = train_run(C, D)

 RUN v4b_continual_s0 | mode continual | 90 phases after phase 0 | device cuda
 phase order: init (random) → hi → zh → ar → hi → zh → ar → hi → ...
 each phase: fresh AdamW (wd 0.1 on matrices), warmup 500 → lr 0.0002 → plateau (<0.5% over 4-eval windows, after 2,000, cap 60,000) → cooldown 2,000 steps to 2e-05
 masked (script-restricted) loss & decoding: True | divergence guard: >1.15x best twice → retry at half lr (max 2)
RESUMING after phase 18 (ar)

  new best checkpoint: hi = phase 10 (chrF 51.5), zh = phase 17 (chrF 25.9), ar = phase 18 (chrF 53.0)
▶ PHASE 19/90 | training hi (993,016 pairs)


[W1015 00:52:38.279842412 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 2048917504 bytes (free: 1124073472, total: 25336807424).
[W1015 00:53:36.087809578 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 1952448512 bytes (free: 1501560832, total: 25336807424).


  [p19 hi] step    500 | lr 2.0e-04 | train 3.678 | val 1.909 (best 1.909) | window Δ n/a | stable | 76k tok/s | phase 1m36s | run ETA: after first phase
  [p19 hi] step  1,000 | lr 2.0e-04 | train 2.815 | val 1.625 (best 1.625) | window Δ n/a | stable | 79k tok/s | phase 3m09s | run ETA: after first phase
  [p19 hi] step  1,500 | lr 2.0e-04 | train 2.650 | val 1.501 (best 1.501) | window Δ n/a | stable | 79k tok/s | phase 4m42s | run ETA: after first phase
  [p19 hi] step  2,000 | lr 2.0e-04 | train 2.521 | val 1.461 (best 1.461) | window Δ n/a | stable | 77k tok/s | phase 6m18s | run ETA: after first phase
  [p19 hi] step  2,500 | lr 2.0e-04 | train 2.480 | val 1.419 (best 1.419) | window Δ n/a | stable | 79k tok/s | phase 7m51s | run ETA: after first phase
  [p19 hi] step  3,000 | lr 2.0e-04 | train 2.454 | val 1.395 (best 1.395) | window Δ n/a | stable | 79k tok/s | phase 9m24s | run ETA: after first phase
  [p19 hi] step  3,500 | lr 2.0e-04 | train 2.414 | val 1.381 (best 1.381) |

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.


  [p87 ar] step    500 | lr 2.0e-04 | train 5.346 | val 3.766 (best 3.766) | window Δ n/a | stable | 71k tok/s | phase 1m36s | run ETA ~2h20m
  [p87 ar] step  1,000 | lr 2.0e-04 | train 4.542 | val 3.494 (best 3.494) | window Δ n/a | stable | 73k tok/s | phase 3m10s | run ETA ~2h20m
  [p87 ar] step  1,500 | lr 2.0e-04 | train 4.394 | val 3.344 (best 3.344) | window Δ n/a | stable | 71k tok/s | phase 4m47s | run ETA ~2h20m
  [p87 ar] step  2,000 | lr 2.0e-04 | train 4.287 | val 3.267 (best 3.267) | window Δ n/a | stable | 73k tok/s | phase 6m21s | run ETA ~2h20m


In [ ]:
print_run_summary(RUN)